We want to detect fast vs slow barcoding using ground truth markings, extract line markings from JPGs, and add metadata from E2Es. 

Fast: `08, 09, 12, 41, 49`

Slow: `17, 23, 35, 36, 47`

So the workflow goes:

```
annotated JPGs
→ extract red/yellow line coordinates
→ create annotation dataframe

E2E files
→ extract patient, B-scan metadata, native image info, layers
→ create E2E dataframe

annotation dataframe + E2E dataframe
→ merge on patient_id + bscan_index
```

The annotation extractor is the first step, and should extract a table with id, scan, structure (EA, barcode), width, and height

Once that is done, it can be merged with metadata, and then follow
```
Original B-scan
↓
EA annotation (green)
Barcode annotation (yellow)
↓
Binary masks
↓
Measurements
```

Test Data:
* the one with no markings is EA8000.jpg
* the one with red markings is EA8011.jpg
* the one with yellow markings is EA8041.jpg
* the one with both markings is EA8043.jpg.

Should work:

whole JPG, crop right OCT panel, HSV threshold, connected components, estimate line endpoints, measure lengths

Once this works then we can merge with metadata

In [ ]:
!pip install opencv-python

In [ ]:
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

PATIENT_ID = "008"

ANNO_DIR = Path("data/heyex/anno") / PATIENT_ID

TEST_FILES = {
    "none": ANNO_DIR / "EA8000.jpg",
    "red_only": ANNO_DIR / "EA8011.jpg",
    "yellow_only": ANNO_DIR / "EA8041.jpg",
    "both": ANNO_DIR / "EA8043.jpg",
}

for expected_label, path in TEST_FILES.items():
    print(
        f"{expected_label:12s} | "
        f"{path.name:12s} | "
        f"exists={path.exists()}"
    )

In [ ]:
def load_image(path: Path) -> np.ndarray:
    """
    Load a JPG using OpenCV.

    Returns
    -------
    np.ndarray
        Image in OpenCV BGR format.
    """
    image = cv2.imread(str(path))

    if image is None:
        raise FileNotFoundError(f"Could not load image: {path}")

    return image


images = {
    expected_label: load_image(path)
    for expected_label, path in TEST_FILES.items()
}

for expected_label, image in images.items():
    height, width = image.shape[:2]

    print(
        f"{expected_label:12s} | "
        f"shape={image.shape} | "
        f"height={height}, width={width}"
    )

In [ ]:
fig, axes = plt.subplots(
    nrows=2,
    ncols=2,
    figsize=(18, 11),
)

axes = axes.ravel()

for ax, (expected_label, image) in zip(axes, images.items()):
    image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    ax.imshow(image_rgb)
    ax.set_title(
        f"{TEST_FILES[expected_label].name}\n"
        f"Expected: {expected_label}"
    )
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# Crop the right OCT panel
# Initial fractional crop boundaries.
#
# Adjust these after inspecting the crop visualization below.
OCT_X_MIN_FRAC = 0.49
OCT_X_MAX_FRAC = 0.995

OCT_Y_MIN_FRAC = 0.07
OCT_Y_MAX_FRAC = 0.82


def crop_oct_panel(
    image: np.ndarray,
    x_min_frac: float = OCT_X_MIN_FRAC,
    x_max_frac: float = OCT_X_MAX_FRAC,
    y_min_frac: float = OCT_Y_MIN_FRAC,
    y_max_frac: float = OCT_Y_MAX_FRAC,
) -> tuple[np.ndarray, dict]:
    """
    Crop the right-side OCT B-scan panel from a HEYEX JPG export.

    Returns
    -------
    crop
        Cropped OCT panel.

    crop_info
        Coordinates of the crop in the original JPG.
    """
    height, width = image.shape[:2]

    x_min = int(round(width * x_min_frac))
    x_max = int(round(width * x_max_frac))
    y_min = int(round(height * y_min_frac))
    y_max = int(round(height * y_max_frac))

    crop = image[y_min:y_max, x_min:x_max].copy()

    crop_info = {
        "x_offset": x_min,
        "y_offset": y_min,
        "x_min": x_min,
        "x_max": x_max,
        "y_min": y_min,
        "y_max": y_max,
        "crop_width": x_max - x_min,
        "crop_height": y_max - y_min,
    }

    return crop, crop_info

In [ ]:
oct_crops = {}
crop_metadata = {}

fig, axes = plt.subplots(
    nrows=2,
    ncols=2,
    figsize=(18, 10),
)

axes = axes.ravel()

for ax, (expected_label, image) in zip(axes, images.items()):
    crop, crop_info = crop_oct_panel(image)

    oct_crops[expected_label] = crop
    crop_metadata[expected_label] = crop_info

    crop_rgb = cv2.cvtColor(crop, cv2.COLOR_BGR2RGB)

    ax.imshow(crop_rgb)
    ax.set_title(
        f"{TEST_FILES[expected_label].name}\n"
        f"OCT crop — expected: {expected_label}"
    )
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
#Define color masks
def create_color_masks(
    oct_crop: np.ndarray,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """
    Create raw red and yellow masks from the cropped OCT panel.

    Returns
    -------
    hsv
        HSV representation of the OCT crop.

    red_mask
        Binary mask for red/magenta annotation pixels.

    yellow_mask
        Binary mask for yellow annotation pixels.
    """
    hsv = cv2.cvtColor(oct_crop, cv2.COLOR_BGR2HSV)

    # Red occupies both ends of OpenCV's hue range.
    red_lower_1 = np.array([0, 90, 80], dtype=np.uint8)
    red_upper_1 = np.array([15, 255, 255], dtype=np.uint8)

    red_lower_2 = np.array([155, 90, 80], dtype=np.uint8)
    red_upper_2 = np.array([179, 255, 255], dtype=np.uint8)

    red_mask_1 = cv2.inRange(
        hsv,
        red_lower_1,
        red_upper_1,
    )

    red_mask_2 = cv2.inRange(
        hsv,
        red_lower_2,
        red_upper_2,
    )

    red_mask = cv2.bitwise_or(
        red_mask_1,
        red_mask_2,
    )

    # Yellow annotation range.
    yellow_lower = np.array([18, 90, 100], dtype=np.uint8)
    yellow_upper = np.array([42, 255, 255], dtype=np.uint8)

    yellow_mask = cv2.inRange(
        hsv,
        yellow_lower,
        yellow_upper,
    )

    return hsv, red_mask, yellow_mask

In [ ]:
raw_masks = {}

for expected_label, crop in oct_crops.items():
    hsv, red_mask, yellow_mask = create_color_masks(crop)

    raw_masks[expected_label] = {
        "red": red_mask,
        "yellow": yellow_mask,
    }

    fig, axes = plt.subplots(
        nrows=1,
        ncols=3,
        figsize=(18, 5),
    )

    axes[0].imshow(
        cv2.cvtColor(crop, cv2.COLOR_BGR2RGB)
    )
    axes[0].set_title(
        f"{TEST_FILES[expected_label].name}\n"
        f"Expected: {expected_label}"
    )

    axes[1].imshow(red_mask, cmap="gray")
    axes[1].set_title("Raw red mask")

    axes[2].imshow(yellow_mask, cmap="gray")
    axes[2].set_title("Raw yellow mask")

    for ax in axes:
        ax.axis("off")

    plt.tight_layout()
    plt.show()

In [ ]:
# Look for line structures
def isolate_horizontal_lines(
    mask: np.ndarray,
    horizontal_kernel_width: int = 15,
) -> np.ndarray:
    """
    Preserve long horizontal colored structures while removing most
    isolated text and JPEG artifacts.
    """
    binary_mask = (mask > 0).astype(np.uint8) * 255

    horizontal_kernel = cv2.getStructuringElement(
        cv2.MORPH_RECT,
        (horizontal_kernel_width, 1),
    )

    horizontal = cv2.morphologyEx(
        binary_mask,
        cv2.MORPH_OPEN,
        horizontal_kernel,
    )

    # Reconnect small breaks introduced by compression or text overlap.
    close_kernel = cv2.getStructuringElement(
        cv2.MORPH_RECT,
        (7, 3),
    )

    horizontal = cv2.morphologyEx(
        horizontal,
        cv2.MORPH_CLOSE,
        close_kernel,
    )

    return horizontal

In [ ]:
def extract_line_components(
    line_mask: np.ndarray,
    color_label: str,
    min_width: int = 20,
    min_area: int = 15,
    min_aspect_ratio: float = 3.0,
) -> list[dict]:
    """
    Extract candidate horizontal measurement-line components.

    Coordinates are relative to the cropped OCT panel.
    """
    num_labels, component_map, stats, centroids = (
        cv2.connectedComponentsWithStats(
            line_mask,
            connectivity=8,
        )
    )

    segments = []

    for component_id in range(1, num_labels):
        x = int(
            stats[component_id, cv2.CC_STAT_LEFT]
        )
        y = int(
            stats[component_id, cv2.CC_STAT_TOP]
        )
        width = int(
            stats[component_id, cv2.CC_STAT_WIDTH]
        )
        height = int(
            stats[component_id, cv2.CC_STAT_HEIGHT]
        )
        area = int(
            stats[component_id, cv2.CC_STAT_AREA]
        )

        aspect_ratio = width / max(height, 1)

        if width < min_width:
            continue

        if area < min_area:
            continue

        if aspect_ratio < min_aspect_ratio:
            continue

        component_pixels = component_map == component_id
        ys, xs = np.where(component_pixels)

        if len(xs) == 0:
            continue

        x_start = int(xs.min())
        x_end = int(xs.max())
        y_mean = float(ys.mean())

        segments.append(
            {
                "color": color_label,
                "component_id": component_id,
                "x_start_crop": x_start,
                "x_end_crop": x_end,
                "y_mean_crop": y_mean,
                "length_px": x_end - x_start + 1,
                "bounding_x": x,
                "bounding_y": y,
                "bounding_width": width,
                "bounding_height": height,
                "pixel_area": area,
                "aspect_ratio": aspect_ratio,
            }
        )

    return segments

In [ ]:
# Run for all four
experiment_results = []
cleaned_masks = {}

for expected_label, crop in oct_crops.items():
    path = TEST_FILES[expected_label]
    crop_info = crop_metadata[expected_label]

    _, red_raw, yellow_raw = create_color_masks(crop)

    red_lines = isolate_horizontal_lines(red_raw)
    yellow_lines = isolate_horizontal_lines(yellow_raw)

    cleaned_masks[expected_label] = {
        "red": red_lines,
        "yellow": yellow_lines,
    }

    red_segments = extract_line_components(
        line_mask=red_lines,
        color_label="red",
    )

    yellow_segments = extract_line_components(
        line_mask=yellow_lines,
        color_label="yellow",
    )

    all_segments = red_segments + yellow_segments

    for segment_index, segment in enumerate(
        all_segments,
        start=1,
    ):
        experiment_results.append(
            {
                "patient_id": PATIENT_ID,
                "filename": path.name,
                "expected_case": expected_label,
                "segment_index": segment_index,
                **segment,
                # Map crop coordinates back to the complete JPG.
                "x_start_jpg": (
                    segment["x_start_crop"]
                    + crop_info["x_offset"]
                ),
                "x_end_jpg": (
                    segment["x_end_crop"]
                    + crop_info["x_offset"]
                ),
                "y_mean_jpg": (
                    segment["y_mean_crop"]
                    + crop_info["y_offset"]
                ),
            }
        )

results_df = pd.DataFrame(experiment_results)

results_df

In [ ]:
summary_rows = []

for expected_label, path in TEST_FILES.items():
    if results_df.empty:
        scan_rows = pd.DataFrame()
    else:
        scan_rows = results_df[
            results_df["filename"] == path.name
        ]

    if scan_rows.empty:
        n_red = 0
        n_yellow = 0
        red_lengths = []
        yellow_lengths = []
    else:
        red_rows = scan_rows[
            scan_rows["color"] == "red"
        ]
        yellow_rows = scan_rows[
            scan_rows["color"] == "yellow"
        ]

        n_red = len(red_rows)
        n_yellow = len(yellow_rows)

        red_lengths = (
            red_rows["length_px"]
            .astype(int)
            .tolist()
        )

        yellow_lengths = (
            yellow_rows["length_px"]
            .astype(int)
            .tolist()
        )

    summary_rows.append(
        {
            "filename": path.name,
            "expected_case": expected_label,
            "detected_red_segments": n_red,
            "detected_yellow_segments": n_yellow,
            "red_lengths_px": red_lengths,
            "yellow_lengths_px": yellow_lengths,
        }
    )

summary_df = pd.DataFrame(summary_rows)

summary_df

In [ ]:
# draw detected segments on crops
def draw_detected_segments(
    crop: np.ndarray,
    segments: list[dict],
) -> np.ndarray:
    """
    Draw detected component endpoints and bounding boxes on the OCT crop.
    """
    output = crop.copy()

    for segment in segments:
        x_start = int(segment["x_start_crop"])
        x_end = int(segment["x_end_crop"])
        y_mean = int(round(segment["y_mean_crop"]))

        box_x = int(segment["bounding_x"])
        box_y = int(segment["bounding_y"])
        box_w = int(segment["bounding_width"])
        box_h = int(segment["bounding_height"])

        # These colors are only for our QC overlay.
        if segment["color"] == "red":
            display_color = (255, 0, 255)
        else:
            display_color = (255, 255, 0)

        cv2.rectangle(
            output,
            (box_x, box_y),
            (box_x + box_w - 1, box_y + box_h - 1),
            display_color,
            thickness=2,
        )

        cv2.circle(
            output,
            (x_start, y_mean),
            radius=4,
            color=display_color,
            thickness=-1,
        )

        cv2.circle(
            output,
            (x_end, y_mean),
            radius=4,
            color=display_color,
            thickness=-1,
        )

        label_text = (
            f"{segment['color']}: "
            f"{segment['length_px']} px"
        )

        cv2.putText(
            output,
            label_text,
            (x_start, max(y_mean - 10, 15)),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.45,
            display_color,
            1,
            cv2.LINE_AA,
        )

    return output

In [ ]:
# inspect original, masks, and final detection
for expected_label, crop in oct_crops.items():
    filename = TEST_FILES[expected_label].name

    if results_df.empty:
        scan_segments = []
    else:
        scan_segments = (
            results_df[
                results_df["filename"] == filename
            ]
            .to_dict(orient="records")
        )

    overlay = draw_detected_segments(
        crop=crop,
        segments=scan_segments,
    )

    fig, axes = plt.subplots(
        nrows=1,
        ncols=4,
        figsize=(22, 5),
    )

    axes[0].imshow(
        cv2.cvtColor(crop, cv2.COLOR_BGR2RGB)
    )
    axes[0].set_title(
        f"{filename}\nOriginal OCT crop"
    )

    axes[1].imshow(
        cleaned_masks[expected_label]["red"],
        cmap="gray",
    )
    axes[1].set_title("Horizontal red mask")

    axes[2].imshow(
        cleaned_masks[expected_label]["yellow"],
        cmap="gray",
    )
    axes[2].set_title("Horizontal yellow mask")

    axes[3].imshow(
        cv2.cvtColor(overlay, cv2.COLOR_BGR2RGB)
    )
    axes[3].set_title("Detected candidate segments")

    for ax in axes:
        ax.axis("off")

    plt.tight_layout()
    plt.show()

In [ ]:
for expected_label, path in TEST_FILES.items():
    print("=" * 70)
    print(f"File:     {path.name}")
    print(f"Expected: {expected_label}")

    if results_df.empty:
        scan_rows = pd.DataFrame()
    else:
        scan_rows = results_df[
            results_df["filename"] == path.name
        ]

    if scan_rows.empty:
        print("Detected segments: none")
        continue

    display_columns = [
        "color",
        "length_px",
        "x_start_crop",
        "x_end_crop",
        "y_mean_crop",
        "bounding_width",
        "bounding_height",
        "aspect_ratio",
    ]

    print(
        scan_rows[display_columns]
        .sort_values(["color", "x_start_crop"])
        .to_string(index=False)
    )